# NexusMart Datathon - Data Profiling and Cleaning

## 1. Objective
This notebook records the initial investigation of the supplied NexusMart raw datasets and establishes evidence for the downstream Python cleaning workflow.

The raw CSVs are treated as read-only. This notebook profiles data and writes baseline audit reports; later cleaning outputs are documented in the processed and rejected data folders and the reports they reference.

## 2. Cleaning principles
- Follow the official NexusMart challenge brief as the primary data specification.
- Investigate duplicate records before removing exact copies.
- Decide missing-value handling by column; do not invent values.
- Validate dates, data types, identifiers, and relationships between tables.
- Preserve rejected records with reason codes when records cannot be reliably corrected.
- Keep core cleaning in Python, not Power Query.
- Keep transformations auditable and leave the supplied raw CSV files unchanged.


In [2]:
from pathlib import Path
import csv
import json
import pandas as pd
from IPython.display import display

# Project paths
PROJECT_ROOT = Path.cwd()

# If VS Code starts the notebook from the notebooks folder,
# move one level up to the project root.
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
REPORT_DIR = PROJECT_ROOT / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data directory exists:", DATA_DIR.exists())
print("CSV files found:", len(list(DATA_DIR.glob("*.csv"))))
print("Pandas version:", pd.__version__)

Project root: c:\Users\LENOVO\Documents\CodeConnect26_DataAnalytics
Data directory exists: True
CSV files found: 9
Pandas version: 2.3.1



## 3. Raw Dataset Inventory

This section records the files supplied with the challenge, their row and
column counts, and their column names before any transformations.
These counts form the baseline for later cleaning and quality reporting.


In [3]:
inventory = []

for file_path in sorted(DATA_DIR.glob("*.csv")):
    df = pd.read_csv(file_path, encoding="utf-8-sig", low_memory=False)

    inventory.append({
        "file": file_path.name,
        "rows": len(df),
        "columns": len(df.columns),
        "column_names": ", ".join(df.columns),
        "memory_mb": round(df.memory_usage(deep=True).sum() / (1024**2), 2),
    })

inventory_df = pd.DataFrame(inventory)

display(
    inventory_df.drop(columns="column_names")
    .sort_values("file")
    .reset_index(drop=True)
)

,file,rows,columns,memory_mb
0,customers.csv,45000,8,16.04
1,locations.csv,650,7,0.15
2,order_items.csv,315143,7,100.96
3,orders.csv,163200,8,59.76
4,payments.csv,168111,6,54.13
5,products.csv,3200,9,1.30
6,returns.csv,16048,6,5.23
7,sellers.csv,420,6,0.10
8,shipping.csv,151228,7,50.48


## 4. Initial Data Quality Assessment

This section checks for missing or blank values and exact duplicate rows across the supplied datasets. These results are an initial assessment; any issues found will be investigated before deciding how to handle them.

In [4]:
quality_results = []

for file_path in sorted(DATA_DIR.glob("*.csv")):
    df = pd.read_csv(file_path, encoding="utf-8-sig", low_memory=False)

    # Count actual missing values (NaN/None)
    missing_by_col = df.isna().sum()

    # Count whitespace-only strings separately
    text_df = df.select_dtypes(include=["object", "string"])
    blank_by_col = text_df.apply(
        lambda col: col.str.strip().eq("").sum()
    )

    # Align blank counts with all columns, including numeric ones
    blank_by_col = blank_by_col.reindex(df.columns, fill_value=0)

    issue_by_col = missing_by_col + blank_by_col

    quality_results.append({
        "file": file_path.name,
        "rows": len(df),
        "exact_duplicate_rows": int(df.duplicated().sum()),
        "columns_with_missing_or_blank": int((issue_by_col > 0).sum()),
        "missing_values": int(missing_by_col.sum()),
        "blank_strings": int(blank_by_col.sum()),
    })

quality_df = pd.DataFrame(quality_results)
display(quality_df)

,file,rows,exact_duplicate_rows,columns_with_missing_or_blank,missing_values,blank_strings
0,customers.csv,45000,0,5,14329,0
1,locations.csv,650,0,3,89,0
2,order_items.csv,315143,0,1,31567,0
3,orders.csv,163200,3200,2,118575,0
4,payments.csv,168111,3296,1,5017,0
5,products.csv,3200,0,4,1343,0
6,returns.csv,16048,0,1,2390,0
7,sellers.csv,420,0,3,89,0
8,shipping.csv,151228,0,3,33929,0


In [5]:
for file_path in sorted(DATA_DIR.glob("*.csv")):
    df = pd.read_csv(file_path, encoding="utf-8-sig", low_memory=False)

    missing = df.isna().sum()
    text_df = df.select_dtypes(include=["object", "string"])
    blanks = text_df.apply(lambda col: col.str.strip().eq("").sum())
    blanks = blanks.reindex(df.columns, fill_value=0)

    issues = pd.DataFrame({
        "missing": missing,
        "blank_strings": blanks,
    })

    issues = issues.loc[issues.sum(axis=1) > 0]

    if not issues.empty:
        print(f"\n--- {file_path.name} ---")
        display(issues)


--- customers.csv ---


,missing,blank_strings
email,1698,0
phone,3599,0
gender,2199,0
city_id,2233,0
birth_year,4600,0



--- locations.csv ---


,missing,blank_strings
province,25,0
latitude,33,0
longitude,31,0



--- order_items.csv ---


...

,missing,blank_strings
return_reason,2390,0



--- sellers.csv ---


,missing,blank_strings
join_date,34,0
seller_tier,22,0
rating,33,0



--- shipping.csv ---


,missing,blank_strings
shipped_date,5943,0
delivered_date,18841,0
shipping_fee,9145,0


## 5. Duplicate Investigation
Exact duplicates are reviewed before any removal. We will compare repeated records with their identifiers and relationships to determine whether they are redundant rows or represent a data issue requiring a different treatment.

In [6]:
for filename in ["orders.csv", "payments.csv"]:
    df = pd.read_csv(DATA_DIR / filename, encoding="utf-8-sig",
                     low_memory=False)

    duplicate_mask = df.duplicated(keep=False)
    duplicate_rows = df.loc[duplicate_mask].sort_values(
        by=list(df.columns)
    )

    print(f"\n--- {filename} ---")
    print("Total rows:", len(df))
    print("Rows belonging to duplicate groups:", int(duplicate_mask.sum()))
    print("Extra copies after keeping one:", int(df.duplicated().sum()))
    print("Columns:", df.columns.tolist())

    if not duplicate_rows.empty:
        display(duplicate_rows.head(12))


--- orders.csv ---
Total rows: 163200
Rows belonging to duplicate groups: 6400
Extra copies after keeping one: 3200
Columns: ['order_id', 'customer_id', 'order_date', 'order_status', 'ship_city_id', 'order_total', 'discount_code', 'channel']


,order_id,customer_id,order_date,order_status,ship_city_id,order_total,discount_code,channel
6068,O0000011,C038724,2024-01-01,Cancelled,32.0,12460,NaN,App
36175,O0000011,C038724,2024-01-01,Cancelled,32.0,12460,NaN,App
2235,O0000075,C033183,2024-01-01,Returned,614.0,1020,NaN,Mobile Web
34692,O0000075,C033183,2024-01-01,Returned,614.0,1020,NaN,Mobile Web
213,O0000086,C034953,2024-01-01,Delivered,461.0,20448,SAVE20,Web
94255,O0000086,C034953,2024-01-01,Delivered,461.0,20448,SAVE20,Web
5528,O0000243,C023249,2024-01-02,Delivered,86.0,3900,NaN,App
107393,O0000243,C023249,2024-01-02,Delivered,86.0,3900,NaN,App
3701,O0000246,C029578,2024-01-02,Delivered,347.0,25501,NaN,Web
49357,O0000246,C029578,2024-01-02,Delivered,347.0,25501,NaN,Web



--- payments.csv ---
Total rows: 168111
Rows belonging to duplicate groups: 6592
Extra copies after keeping one: 3296
Columns: ['payment_id', 'order_id', 'payment_method', 'payment_status', 'amount', 'payment_date']


,payment_id,order_id,payment_method,payment_status,amount,payment_date
1771,PAY00000011,O0000011,COD,Cancelled,12460,2024-01-06
107842,PAY00000011,O0000011,COD,Cancelled,12460,2024-01-06
3440,PAY00000060,O0000059,cod,Paid,79779,2024-01-04
73500,PAY00000060,O0000059,cod,Paid,79779,2024-01-04
4435,PAY00000131,O0000130,BANK TRANSFER,Paid,1814,2024-06-31
139052,PAY00000131,O0000130,BANK TRANSFER,Paid,1814,2024-06-31
6886,PAY00000155,O0000153,Card,Paid,7930,Jan 1 2024
35303,PAY00000155,O0000153,Card,Paid,7930,Jan 1 2024
3417,PAY00000188,O0000186,Bank Transfer,Success,26351,02/01/2024
100680,PAY00000188,O0000186,Bank Transfer,Success,26351,02/01/2024


**Interpretation:** this cell is a diagnostic of duplicate groups in the raw exports; it does not remove rows. The later exact-duplicate removal is evidenced by `reports/duplicate_removal_log.csv`: 3,200 copies from `orders.csv` and 3,296 from `payments.csv` were removed from the processed path. The raw files remain unchanged.


In [7]:
display(inventory_df[["file", "column_names"]])

quality_df.to_csv(
    REPORT_DIR / "notebook_baseline_quality.csv",
    index=False
)

print("Baseline report saved:", REPORT_DIR / "notebook_baseline_quality.csv")

,file,column_names
0,customers.csv,"customer_id, full_name, email, phone, gender, ..."
1,locations.csv,"city_id, city, province, region, latitude, lon..."
2,order_items.csv,"order_item_id, order_id, product_id, seller_id..."
3,orders.csv,"order_id, customer_id, order_date, order_statu..."
4,payments.csv,"payment_id, order_id, payment_method, payment_..."
5,products.csv,"product_id, product_name, category, sub_catego..."
6,returns.csv,"return_id, order_item_id, return_date, return_..."
7,sellers.csv,"seller_id, seller_name, join_date, seller_tier..."
8,shipping.csv,"shipment_id, order_id, courier, shipped_date, ..."


Baseline report saved: c:\Users\LENOVO\Documents\CodeConnect26_DataAnalytics\reports\notebook_baseline_quality.csv


## 6. Identifier integrity

The next audit checks candidate primary keys after excluding exact duplicate copies in a temporary in-memory view. It does not edit or remove rows from the raw files. Results are written to `reports/identifier_audit.csv`.


In [8]:
key_columns = {
    "customers.csv": ["customer_id"],
    "products.csv": ["product_id"],
    "sellers.csv": ["seller_id"],
    "locations.csv": ["city_id"],
    "orders.csv": ["order_id"],
    "order_items.csv": ["order_item_id"],
    "payments.csv": ["payment_id"],
    "shipping.csv": ["shipment_id"],
    "returns.csv": ["return_id"],
}

key_audit = []

for filename, keys in key_columns.items():
    df = pd.read_csv(DATA_DIR / filename, encoding="utf-8-sig",
                     low_memory=False)

    # Remove only exact duplicate copies in this temporary analysis.
    # The raw files remain unchanged.
    df_unique = df.drop_duplicates()

    for key in keys:
        if key not in df_unique.columns:
            key_audit.append({
                "file": filename,
                "key": key,
                "status": "Column not found",
                "duplicate_key_rows": None,
                "conflicting_keys": None,
            })
            continue

        nonblank = df_unique[key].notna()
        repeated = df_unique.loc[nonblank & df_unique[key].duplicated(False)]

        conflicts = 0
        if not repeated.empty:
            conflicts = int(
                (repeated.groupby(key, dropna=False)
                 .apply(lambda group: len(group.drop_duplicates()) > 1,
                        include_groups=False)).sum()
            )

        key_audit.append({
            "file": filename,
            "key": key,
            "status": "Checked",
            "duplicate_key_rows": len(repeated),
            "conflicting_keys": conflicts,
        })

key_audit_df = pd.DataFrame(key_audit)
display(key_audit_df)

,file,key,status,duplicate_key_rows,conflicting_keys
0,customers.csv,customer_id,Checked,0,0
1,products.csv,product_id,Checked,0,0
2,sellers.csv,seller_id,Checked,0,0
3,locations.csv,city_id,Checked,0,0
4,orders.csv,order_id,Checked,0,0
5,order_items.csv,order_item_id,Checked,0,0
6,payments.csv,payment_id,Checked,0,0
7,shipping.csv,shipment_id,Checked,0,0
8,returns.csv,return_id,Checked,0,0


## 7. Date quality
Inspect date formats and identify values that cannot be parsed or fall outside the expected business period. Invalid dates will be investigated rather than silently replaced.

In [9]:
date_columns = {
    "orders.csv": ["order_date"],
    "payments.csv": ["payment_date"],
    "shipping.csv": ["shipped_date", "delivered_date"],
    "returns.csv": ["return_date"],
    "sellers.csv": ["join_date"],
}

date_audit = []

for filename, columns in date_columns.items():
    df = pd.read_csv(DATA_DIR / filename, encoding="utf-8-sig",
                     low_memory=False)

    for col in columns:
        if col not in df.columns:
            continue

        raw = df[col].astype("string").str.strip()
        present = raw.notna() & raw.ne("")

        parsed = pd.to_datetime(
            raw,
            format="mixed",
            dayfirst=True,
            errors="coerce"
        )

        invalid = present & parsed.isna()

        date_audit.append({
            "file": filename,
            "column": col,
            "missing_values": int((~present).sum()),
            "unparseable_dates": int(invalid.sum()),
            "earliest_parsed_date": parsed.min(),
            "latest_parsed_date": parsed.max(),
        })

        if invalid.any():
            print(f"\nUnparseable examples: {filename} — {col}")
            display(df.loc[invalid, [col]].head(10))

date_audit_df = pd.DataFrame(date_audit)
display(date_audit_df)


Unparseable examples: orders.csv — order_date


,order_date
52,00/00/2024
80,00/00/2024
579,2024-02-30
770,2025-04-31
969,2024-04-31
1166,2024-04-31
1591,2024-04-31
1711,00/00/2024
1752,2024-02-30
1780,2024-04-31



Unparseable examples: payments.csv — payment_date


,payment_date
319,2024-02-30
373,00/00/2025
754,2025-06-31
941,00/00/2024
1058,2025-02-30
1164,2024-06-31
1228,2024-04-31
1394,00/00/2025
1825,2024-02-30
1930,00/00/2025



Unparseable examples: shipping.csv — shipped_date


...


Unparseable examples: returns.csv — return_date


,return_date
15,00/00/2024
53,2024-02-30
97,2024-02-30
665,2024-02-30
787,00/00/2024
818,2024-02-30
868,2024-06-31
1031,00/00/2024
1646,2024-02-30
1803,2024-04-31



Unparseable examples: sellers.csv — join_date


,join_date
69,2020-06-31
112,2020-06-31
146,00/00/2021
221,2022-04-31


,file,column,missing_values,unparseable_dates,earliest_parsed_date,latest_parsed_date
0,orders.csv,order_date,0,911,2024-01-01,2027-09-07
1,payments.csv,payment_date,5017,871,2024-01-01,2025-07-18
2,shipping.csv,shipped_date,5943,819,2024-01-01,2025-07-02
3,shipping.csv,delivered_date,18841,760,2024-01-02,2025-09-26
4,returns.csv,return_date,0,106,2023-08-18,2025-06-30
5,sellers.csv,join_date,34,4,2019-11-27,2023-12-01


In [10]:
key_audit_df.to_csv(REPORT_DIR / "identifier_audit.csv", index=False)
date_audit_df.to_csv(REPORT_DIR / "date_audit.csv", index=False)

print("Saved identifier and date audit reports.")

Saved identifier and date audit reports.


## 8. Detailed inspection of individual datasets

In [11]:
# Inspect orders first because order dates and duplicates affect revenue.

df = pd.read_csv(DATA_DIR / "orders.csv")

display(df.head())
df.info()

print("Shape:", df.shape)
print("\nMissing values:")
display(df.isnull().sum())

print("\nExact duplicate rows:", df.duplicated().sum())

print("\nOrder status values:")
display(df["order_status"].value_counts(dropna=False))

print("\nOrder date examples:")
display(df["order_date"].value_counts().head(15))

,order_id,customer_id,order_date,order_status,ship_city_id,order_total,discount_code,channel
0,O0106517,C042364,2025-02-04,DELIVERD,316.0,5206,NaN,Website
1,O0129677,C022887,2025-04-06,Delivered,347.0,675,NaN,App
2,O0083748,C010022,Nov 23 2024,Delivered,281.0,2362,NaN,Web
3,O0027756,C032262,May 2 2024,Delivered,347.0,5858,NaN,App
4,O0119576,C035932,2025-03-14,Completed,450.0,35935,SAVE25,Mobile Web


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 163200 entries, 0 to 163199
Data columns (total 8 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   order_id       163200 non-null  object 
 1   customer_id    163200 non-null  object 
 2   order_date     163200 non-null  object 
 3   order_status   163200 non-null  object 
 4   ship_city_id   156684 non-null  float64
 5   order_total    163200 non-null  object 
 6   discount_code  51141 non-null   object 
 7   channel        163200 non-null  object 
dtypes: float64(1), object(7)
memory usage: 10.0+ MB
Shape: (163200, 8)

Missing values:


order_id              0
customer_id           0
order_date            0
order_status          0
ship_city_id       6516
order_total           0
discount_code    112059
channel               0
dtype: int64


Exact duplicate rows: 3200

Order status values:


order_status
Delivered     108289
Cancelled      11222
Complete        7292
DELIVERD        7175
Completed       7153
DELIVERED       7118
delivered       6911
Returned        2194
In Transit      1024
cancelled        971
Canceled         948
Cancelled        924
CANCELLED        907
returned         279
RETURNED         228
Return           210
In-Transit        97
in transit        94
IN TRANSIT        88
Shipped           76
Name: count, dtype: int64


Order date examples:


order_date
2025-05-31    678
2024-11-11    658
2025-06-01    644
2025-03-30    591
2025-06-03    580
2025-03-25    569
2025-06-05    561
2024-11-10    556
2025-03-29    556
2025-06-06    549
2025-03-24    541
2024-11-29    535
2025-03-28    503
2024-04-07    481
2024-11-09    477
Name: count, dtype: int64

## 9. Cleaning milestone handoff

This notebook contains the executed raw profiling and investigation cells above. It does not contain the later date, sentinel, and invalid-order-item quarantine transformations. Those completed operations are evidenced by the processed/rejected files and reports listed below; the current repository does not contain the original transformation cells or scripts for every earlier cleaning pass. Do not treat this notebook as a full raw-to-processed rebuild.

Key audit outputs are `reports/notebook_baseline_quality.csv`, `reports/duplicate_removal_log.csv`, `reports/date_quarantine_log.csv`, `reports/sentinel_quarantine_log.csv`, and `reports/invalid_order_items_quarantine_log.csv`. Final reconciliation and remaining decisions are summarized in `reports/consolidated_quality_report.csv`, `reports/final_cleaning_decision_log.csv`, and `reports/cleaning_validation_report.csv`.

For a read-only summary of the final milestone, see `02_data_cleaning_validation.ipynb`. That notebook reads reports only. `src/finalize_cleaning.py` is an in-place finalization step for an already prepared `data/processed/` directory, not a rebuild from raw data.
